In [ ]:
import pyspark.sql.functions as F

In [ ]:
catalog = dbutils.widgets.get("catalog")
pipeline_id = dbutils.widgets.get("pipeline_id")
run_id = dbutils.widgets.get("run_id")
task_id = dbutils.widgets.get("task_id")
processing_date = dbutils.widgets.get("processing_date")

In [ ]:
landing_path = f"/Volumes/{catalog}/landing/football_data/"

In [ ]:
columns_map = F.create_map(
    F.lit("pipeline_id"), F.lit(pipeline_id),
    F.lit("run_id"), F.lit(run_id),
    F.lit("task_id"), F.lit(task_id),
    F.lit("processing_date"), F.lit(processing_date)
)

In [ ]:
tables = ["scorers", "standings", "matches", "teams"]
for table in tables:
    df = spark.read.format("json").option("multiline", "true").load(landing_path + table + "/*/")
    df = df.withColumn(
        "_metadata", columns_map
    )
    df.write.format("delta").mode("overwrite").saveAsTable(f"{catalog}.bronze.raw_{table}_tbl")